# Lab 2 Part 3: image agents

The first code cell creates a pinned environment and prints its commit, versions,
device and seed. It does not replace packages inside the running notebook kernel.
Windows, Linux, Colab and Kaggle paths are handled without a hard-coded `/content`
directory on a PC. First-time setup needs internet and several GB of free disk.

Use a fresh kernel and Run All. The default is a full run. `LAB2_SMOKE_ONLY=1`
is only for developer checks and is clearly printed; those outputs are not a
completed GPU experiment. Keep your real executed copy after the full run.
Run only one GPU training/generation notebook at a time. Setup is locked, but
two concurrent large models can still exceed the GPU's memory.

Run Part 2 first for the domain-style adapter. This notebook uses a clearly marked synthetic image to check wiring. Replace `INPUT_IMAGE` with your chosen lab test images for the real evaluation; one synthetic input is not the required diverse test set.

In [1]:
from pathlib import Path
import os, subprocess, sys, platform, json, time

os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "120")
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "60")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

SMOKE_ONLY = os.environ.get("LAB2_SMOKE_ONLY", "0") == "1"
roots = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in roots if (p / "tools/setup_runtime.py").exists()), None)
if ROOT is None:
    if os.environ.get("LAB2_ROOT"):
        ROOT = Path(os.environ["LAB2_ROOT"]).expanduser().resolve()
    elif Path("/content").exists():
        ROOT = Path("/content/Lab-2")
    elif Path("/kaggle/working").exists():
        ROOT = Path("/kaggle/working/Lab-2")
    else:
        ROOT = Path.home() / "DATA266" / "Lab-2"
    if not ROOT.exists():
        ROOT.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "clone", "https://github.com/denishatank12/Lab-2.git", str(ROOT)], check=True)
    if not (ROOT / "tools/setup_runtime.py").exists():
        raise RuntimeError(f"{ROOT} is an older/incomplete checkout. Update it with git pull, then restart this notebook.")
print("Notebook host:", platform.python_version(), "repository:", ROOT)
print("Mode:", "SMOKE ONLY: no full training or model-backed evaluation" if SMOKE_ONLY else "FULL RUN")
# Refresh a clean main checkout, but never overwrite local edits or reset a branch.
if not SMOKE_ONLY and os.environ.get("LAB2_UPDATE_REPOSITORY", "1") == "1":
    branch = subprocess.check_output(["git", "-C", str(ROOT), "branch", "--show-current"], text=True).strip()
    dirty = subprocess.check_output(["git", "-C", str(ROOT), "status", "--porcelain"], text=True).strip()
    if branch == "main" and not dirty:
        subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only"], check=True)
    else:
        print("Kept existing checkout: local edits or a non-main branch. No files were reset.")
if "SETUP_SCHEMA = 2" not in (ROOT / "tools/setup_runtime.py").read_text(encoding="utf-8"):
    # A freshly downloaded notebook must not silently execute the old broken installer.
    previous_root = ROOT
    ROOT = ROOT.with_name(ROOT.name + "-notebook-v2")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "https://github.com/denishatank12/Lab-2.git", str(ROOT)], check=True)
    if not (ROOT / "tools/setup_runtime.py").is_file() or "SETUP_SCHEMA = 2" not in (ROOT / "tools/setup_runtime.py").read_text(encoding="utf-8"):
        raise RuntimeError(f"Update the checkout at {ROOT} before running. The original checkout was preserved.")
    print("Using repaired code at", ROOT, "; the older checkout and its results remain at", previous_root)
subprocess.run([sys.executable, str(ROOT / "tools/setup_runtime.py"),
                "--device", "cpu" if SMOKE_ONLY else os.environ.get("LAB2_DEVICE", "cuda")], check=True)
requested_device = "cpu" if SMOKE_ONLY else os.environ.get("LAB2_DEVICE", "cuda")
import shutil
use_cuda = requested_device == "cuda" or (requested_device == "auto" and shutil.which("nvidia-smi") is not None)
runtime_file = ROOT / (".lab2_runtime_cuda.json" if use_cuda else ".lab2_runtime_cpu.json")
PYTHON = Path(json.loads(runtime_file.read_text())["python"])
SEED = 266
print("All remaining commands use:", PYTHON)

def run_script(relative, *arguments):
    command = [str(PYTHON), "-u", str(ROOT / relative), *map(str, arguments)]
    logs = ROOT / "runtime_checks"
    logs.mkdir(exist_ok=True)
    log = logs / (Path(relative).stem + ".log")
    print("Running", Path(relative).name, flush=True)
    with log.open("a", encoding="utf-8") as handle:
        process = subprocess.Popen(command, cwd=ROOT, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace")
        for line in process.stdout:
            print(line, end="", flush=True)
            handle.write(line)
            handle.flush()
        code = process.wait()
    if code:
        raise RuntimeError(f"{Path(relative).name} stopped (exit {code}). Full diagnostic log: {log}")


Notebook host: 3.12.6 repository: /Users/denishatank/Documents/ChatGPT/Gen AI/Lab-2
Mode: SMOKE ONLY: no full training or model-backed evaluation


{
  "commit": "3b12f56",
  "working_tree_dirty": true,
  "python": "3.12.6",
  "seed": 266,
  "executable": "/Users/denishatank/Documents/ChatGPT/Gen AI/Lab-2/.venv/bin/python",
  "versions": {
    "torch": "2.5.1",
    "torchvision": "0.20.1",
    "diffusers": "0.31.0",
    "transformers": "4.46.2",
    "peft": "0.13.2",
    "datasets": "3.1.0",
    "huggingface-hub": "0.26.2"
  },
  "cuda_available": false,
  "gpu": "cpu"
}
PASS: core imports, matching torchvision kernels and device preflight


LAB2_PYTHON=/Users/denishatank/Documents/ChatGPT/Gen AI/Lab-2/.venv/bin/python
All remaining commands use: /Users/denishatank/Documents/ChatGPT/Gen AI/Lab-2/.venv/bin/python


## Smoke and red-team checks
Mocked model calls exercise routing, missing/corrupt inputs, bounded revisions and generation exceptions. These tests are not model-quality or human-evaluation scores.

In [2]:
run_script("part3_agents/src/smoke_test.py")
run_script("part3_agents/src/run_red_team.py", "--output", ROOT / "runtime_checks/red_team.json")
if not SMOKE_ONLY:
    run_script("tools/preflight.py", "--require-cuda")

Running smoke_test.py


PASS: safe routing, corrupt files, real image validation, bounded revisions, unique outputs and error recovery


Running run_red_team.py


{


  "cases": 10,


  "passed": 10


}


## Choose an image
Provide a real file path with `LAB2_INPUT_IMAGE`, or replace `INPUT_IMAGE` below. The synthetic fallback is only a functional demonstration.

In [3]:
P3 = ROOT / "part3_agents"
INPUT_IMAGE = Path(os.environ.get("LAB2_INPUT_IMAGE", str(P3 / "sample_input.png")))
if not INPUT_IMAGE.exists() and "LAB2_INPUT_IMAGE" not in os.environ:
    run_script("tools/make_sample_image.py", INPUT_IMAGE)
    print("Using SYNTHETIC DEBUG IMAGE, not the final evaluation set.")
if not INPUT_IMAGE.is_file():
    raise FileNotFoundError(f"Input image does not exist: {INPUT_IMAGE}")
print("Input:", INPUT_IMAGE)

Input: /Users/denishatank/Documents/ChatGPT/Gen AI/Lab-2/part3_agents/sample_input.png


## Compare with and without critique
Captioning and visual QA run on CPU; a single img2img pipeline is offloaded for the RTX 4050. Every scenario and revision gets a separate image path. Generation failures are saved as errors, not marked successful. There are at most two revisions.

In [4]:
ADAPTER = ROOT / "part2_diffusion/outputs/run1"
if not SMOKE_ONLY and not (ADAPTER / "pytorch_lora_weights.safetensors").is_file():
    raise RuntimeError("Finish Part 2 run1 first, or point ADAPTER to a completed adapter folder.")
for filename, without in [("scenario_traces_no_critique.json", True), ("scenario_traces.json", False)]:
    arguments = ["--image", INPUT_IMAGE, "--output", P3 / "outputs" / filename]
    if without:
        arguments.append("--no-critique")
    if not SMOKE_ONLY:
        arguments.extend(["--enable-models", "--lora-path", ADAPTER])
    run_script("part3_agents/src/run_scenarios.py", *arguments)
print("Traces saved. Judge task success and critic agreement using your real scenario rubrics.")

Running run_scenarios.py


S01 models_disabled 


S02 unsupported 


S03 models_disabled 


S04 models_disabled 


S05 models_disabled 


... (41 outputs left out)

  "note": "Task success and critic agreement require the two members to judge the saved traces.",


  "output": "/Users/denishatank/Documents/ChatGPT/Gen AI/Lab-2/part3_agents/outputs/scenario_traces.json"


}


Traces saved. Judge task success and critic agreement using your real scenario rubrics.


## Optional local demo app
Launch only when you want the interactive demo. It is disabled by default so Run All can finish. The server stays local; public sharing is not enabled.

In [5]:
LAUNCH_APP = False
if LAUNCH_APP:
    app_process = subprocess.Popen([str(PYTHON), str(P3 / "src/app.py")], cwd=ROOT)
    print("Local app started. Stop app_process when finished.")
else:
    print("App not launched. Run the app.py command when ready for the demo.")

App not launched. Run the app.py command when ready for the demo.


## Limits
Visible text is currently a VQA guess, not verified OCR. The image critic uses CLIP and a generated caption, not independent human judgment. Text guardrails and the diffusion safety checker reduce risk but are not a guarantee. Full grading also requires real test images, human judgments, team documentation and model uploads.